In [1]:
# ============================================================
# COMPANY BANGLA PLATE MODEL PERFORMANCE TEST
#
# Purpose:
# Test the fine-tuned Bangla license-plate token model on the
# company's real plate-image dataset.
#
# IMPORTANT:
# The company ZIP contains plate images only and no ground-truth
# labels. Therefore this script cannot calculate true OCR
# accuracy, exact-match accuracy, precision, recall, or F1.
#
# It reports:
#   1. Number of company plate images tested
#   2. Images with detected tokens
#   3. Images with no detection
#   4. Average detected tokens per image
#   5. Average token confidence
#   6. Average inference time
#   7. Reconstructed token sequence for every image
#   8. Annotated prediction images
# ============================================================
import subprocess
import sys

def pip(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True
    )

pip("ultralytics==8.3.*")

print("Installation complete.")
from pathlib import Path
import time

import cv2
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt



# ============================================================
# 1. CONFIGURATION
# ============================================================

# IMPORTANT:
# Set this to the fine-tuned model you want to test.
# Example:
# MODEL_PATH = Path("/kaggle/input/my-model/best.pt")
print("Searching for extracted best.pt contents...")

for p in Path("/kaggle/input").rglob("data.pkl"):
    print(p)



data_files = list(
    Path("/kaggle/input").rglob("data.pkl")
)

best_dirs = []

for p in data_files:
    if p.parent.name == "best":
        best_dirs.append(p.parent)

if len(best_dirs) == 0:
    raise FileNotFoundError(
        "Could not find the extracted best model."
    )

if len(best_dirs) > 1:
    print("Multiple model folders found:")
    for p in best_dirs:
        print(p)

best_dir = best_dirs[0]

print("Using model folder:")
print(best_dir)

NEW_MODEL_PATH = Path(
    "/kaggle/working/best.pt"
)

with zipfile.ZipFile(
    NEW_MODEL_PATH,
    "w",
    compression=zipfile.ZIP_STORED
) as z:

    for file_path in best_dir.rglob("*"):

        if file_path.is_file():

            relative = file_path.relative_to(
                best_dir
            )

            archive_name = (
                Path("best") / relative
            )

            z.write(
                file_path,
                archive_name.as_posix()
            )

print("\nCreated:")
print(NEW_MODEL_PATH)

print(
    "Size:",
    NEW_MODEL_PATH.stat().st_size,
    "bytes"
)


model = YOLO(
    "/kaggle/working/best.pt"
)

print("===================================")
print("NEW FINE-TUNED MODEL LOADED")
print("===================================")

print(model.names)

NEW_MODEL_PATH = Path(
    "/kaggle/working/best.pt"
)

# ============================================================

def find_new_model():

    if NEW_MODEL_PATH.exists():
        return NEW_MODEL_PATH

    print(
        "Configured NEW_MODEL_PATH was not found."
    )

    print(
        "\nSearching /kaggle/input for "
        "bangla_lpdb_yolo11m_motion_finetune/weights/best.pt ..."
    )

    matches = list(
        Path("/kaggle/input").rglob(
            "bangla_lpdb_yolo11m_motion_finetune/weights/best.pt"
        )
    )

    if len(matches) == 1:
        print(
            "Found new model:",
            matches[0]
        )
        return matches[0]

    if len(matches) > 1:
        print(
            "\nMultiple new models found:"
        )

        for p in matches:
            print(p)

        raise RuntimeError(
            "Set NEW_MODEL_PATH manually because "
            "multiple fine-tuned models were found."
        )

    raise FileNotFoundError(
        "\nNew fine-tuned best.pt was not found.\n"
        "Change NEW_MODEL_PATH to the path of your "
        "saved fine-tuned best.pt."
    )


NEW_MODEL_PATH = find_new_model()

if not NEW_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"New fine-tuned best.pt not found:\n{NEW_MODEL_PATH}"
    )

print("\nNEW MODEL:")
print(NEW_MODEL_PATH)

# IMPORTANT:
# Set this to the Kaggle input folder containing the company's
# plate images.
# Example:
# COMPANY_ROOT = Path("/kaggle/input/company-plate-images")
COMPANY_ROOT = Path("/kaggle/input/datasets/tanvirmahtabtaneem/number-plate-test-images/images")

OUTPUT_ROOT = Path(
    "/kaggle/working/company_plate_test"
)

ANNOTATED_DIR = OUTPUT_ROOT / "annotated"
RESULT_CSV = OUTPUT_ROOT / "company_plate_results.csv"
SUMMARY_CSV = OUTPUT_ROOT / "company_plate_summary.csv"

CONF_THRESHOLD = 0.25
IMG_SIZE = 960
DEVICE = 0
MAX_IMAGES = None
MAX_ANNOTATED_IMAGES = 50

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ============================================================
# 2. FIND MODEL
# ============================================================


MODEL_PATH=NEW_MODEL_PATH
print("=" * 70)
print("MODEL")
print("=" * 70)
print(MODEL_PATH)


# ============================================================
# 3. FIND COMPANY IMAGES
# ============================================================

def find_company_images():

    if COMPANY_ROOT is None:
        raise RuntimeError(
            "Set COMPANY_ROOT to the Kaggle input folder containing "
            "the company's plate images."
        )

    root = Path(COMPANY_ROOT)

    if not root.exists():
        raise FileNotFoundError(
            f"Company image folder not found:\n{root}"
        )

    image_files = []

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            image_files.append(path)

    image_files = sorted(image_files)

    if MAX_IMAGES is not None:
        image_files = image_files[:MAX_IMAGES]

    if len(image_files) == 0:
        raise RuntimeError(
            "No plate images were found under COMPANY_ROOT."
        )

    return image_files


IMAGE_FILES = find_company_images()

print()
print("=" * 70)
print("COMPANY DATASET")
print("=" * 70)
print("Images found:", len(IMAGE_FILES))

for path in IMAGE_FILES[:10]:
    print(path)


# ============================================================
# 4. LOAD MODEL
# ============================================================

print()
print("=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = YOLO(str(MODEL_PATH))

print("Model loaded successfully.")
print("Number of classes:", len(model.names))


# ============================================================
# 5. OUTPUT FOLDERS
# ============================================================

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

ANNOTATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 6. ORDER DETECTED TOKENS
# ============================================================

def order_tokens(detections):

    if not detections:
        return []

    detections.sort(
        key=lambda x: x["cy"]
    )

    rows = []

    for item in detections:

        placed = False

        for row in rows:

            row_y = sum(
                x["cy"] for x in row
            ) / len(row)

            row_height = sum(
                x["height"] for x in row
            ) / len(row)

            tolerance = 0.55 * row_height

            if abs(item["cy"] - row_y) <= tolerance:
                row.append(item)
                placed = True
                break

        if not placed:
            rows.append([item])

    rows.sort(
        key=lambda row: sum(
            x["cy"] for x in row
        ) / len(row)
    )

    for row in rows:
        row.sort(
            key=lambda x: x["cx"]
        )

    ordered = []

    for row in rows:
        ordered.extend(row)

    return ordered


# ============================================================
# 7. PREDICT ONE PLATE IMAGE
# ============================================================

def predict_plate(image):

    results = model.predict(
        source=image,
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )

    result = results[0]

    if result.boxes is None:
        return [], result

    detections = []

    for index in range(len(result.boxes)):

        confidence = float(
            result.boxes.conf[index].item()
        )

        if confidence < CONF_THRESHOLD:
            continue

        class_id = int(
            result.boxes.cls[index].item()
        )

        if class_id not in model.names:
            continue

        x1, y1, x2, y2 = (
            result.boxes.xyxy[index]
            .cpu()
            .numpy()
            .tolist()
        )

        detections.append({
            "class_id": class_id,
            "name": model.names[class_id],
            "confidence": confidence,
            "cx": (x1 + x2) / 2.0,
            "cy": (y1 + y2) / 2.0,
            "width": x2 - x1,
            "height": y2 - y1,
        })

    ordered = order_tokens(detections)

    return ordered, result


# ============================================================
# 8. RUN TEST
# ============================================================

all_rows = []
total_time = 0.0
detected_images = 0
no_detection_images = 0
total_tokens = 0
all_confidences = []

print()
print("=" * 70)
print("STARTING COMPANY MODEL TEST")
print("=" * 70)

for index, image_path in enumerate(IMAGE_FILES):

    image = cv2.imread(str(image_path))

    if image is None:
        print("Could not read:", image_path)
        continue

    start_time = time.perf_counter()

    detections, result = predict_plate(image)

    elapsed = time.perf_counter() - start_time
    total_time += elapsed

    token_names = [
        item["name"]
        for item in detections
    ]

    token_sequence = " | ".join(token_names)
    token_count = len(detections)

    if token_count > 0:
        detected_images += 1
    else:
        no_detection_images += 1

    total_tokens += token_count

    confidences = [
        item["confidence"]
        for item in detections
    ]

    all_confidences.extend(confidences)

    average_confidence = 0.0

    if confidences:
        average_confidence = sum(confidences) / len(confidences)

    # Save annotated prediction images for the first N images.
    if index < MAX_ANNOTATED_IMAGES:
        annotated = result.plot()
        output_image = ANNOTATED_DIR / image_path.name
        cv2.imwrite(str(output_image), annotated)

    all_rows.append({
        "image": image_path.name,
        "image_path": str(image_path),
        "detected_tokens": token_count,
        "token_sequence": token_sequence,
        "average_confidence": average_confidence,
        "inference_time_ms": elapsed * 1000.0,
        "status": "DETECTED" if token_count > 0 else "NO_DETECTION",
    })

    if (index + 1) % 25 == 0 or index == len(IMAGE_FILES) - 1:
        print(
            f"{index + 1}/{len(IMAGE_FILES)} images processed"
        )


# ============================================================
# 9. SAVE DETAILED RESULTS
# ============================================================

results_df = pd.DataFrame(all_rows)

results_df.to_csv(
    RESULT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. SUMMARY METRICS
# ============================================================

image_count = len(results_df)

detection_rate = 0.0
if image_count > 0:
    detection_rate = detected_images / image_count * 100.0

no_detection_rate = 0.0
if image_count > 0:
    no_detection_rate = no_detection_images / image_count * 100.0

average_tokens = 0.0
if image_count > 0:
    average_tokens = total_tokens / image_count

average_confidence = 0.0
if all_confidences:
    average_confidence = sum(all_confidences) / len(all_confidences)

average_time_ms = 0.0
if image_count > 0:
    average_time_ms = total_time / image_count * 1000.0

summary_df = pd.DataFrame([{
    "total_images": image_count,
    "images_with_detection": detected_images,
    "images_without_detection": no_detection_images,
    "detection_rate_percent": detection_rate,
    "no_detection_rate_percent": no_detection_rate,
    "average_tokens_per_image": average_tokens,
    "average_token_confidence": average_confidence,
    "average_inference_time_ms": average_time_ms,
    "total_inference_time_sec": total_time,
}])

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 11. PRINT SUMMARY
# ============================================================

print()
print("=" * 70)
print("COMPANY MODEL PERFORMANCE SUMMARY")
print("=" * 70)

print("Total images:", image_count)
print("Images with detection:", detected_images)
print("Images without detection:", no_detection_images)
print(f"Detection rate: {detection_rate:.2f}%")
print(f"No-detection rate: {no_detection_rate:.2f}%")
print(f"Average tokens per image: {average_tokens:.2f}")
print(f"Average token confidence: {average_confidence:.4f}")
print(f"Average inference time: {average_time_ms:.2f} ms")
print(f"Total inference time: {total_time:.2f} sec")


# ============================================================
# 12. SHOW SAMPLE PREDICTIONS
# ============================================================

print()
print("=" * 70)
print("SAMPLE PREDICTIONS")
print("=" * 70)

for _, row in results_df.head(20).iterrows():

    print()
    print("Image:", row["image"])
    print("Tokens:", row["token_sequence"])
    print(
        "Confidence:",
        f'{row["average_confidence"]:.4f}'
    )
    print(
        "Time:",
        f'{row["inference_time_ms"]:.2f} ms'
    )
    print("Status:", row["status"])


# ============================================================
# 13. FINAL OUTPUT LOCATIONS
# ============================================================

print()
print("=" * 70)
print("TEST COMPLETE")
print("=" * 70)
print("Detailed results:", RESULT_CSV)
print("Summary:", SUMMARY_CSV)
print("Annotated images:", ANNOTATED_DIR)
print()
print("IMPORTANT:")
print(
    "The company dataset has no ground-truth labels, so this test "
    "cannot calculate true OCR accuracy. It measures the model's "
    "actual predictions, detection coverage, confidence, and speed "
    "on the company's images."
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:00
Installation complete.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Searching for extracted best.pt contents...
/kaggle/input/datasets/tanvirmahtabtaneem/best-zip/best/data.pkl
Using model folder:
/kaggle/input/datasets/tanvirmahtabtaneem/best-zip/best

Created:
/kaggle/working/best.pt
Size: 40696293 bytes
NEW FINE-TUNED MODEL LOADED
{0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: 'Metro', 11: 'A', 12: 'Bha', 13: 'Cha', 14: 'Chha', 15: 'Da', 16: 'DA', 17: 'E', 18: 'Ga', 19: 'Gha', 20: 'Ha', 21: 'Ja', 22: 'Jha', 23: 'Ka', 24: 'Kha', 25: 'La', 26: 'Ma', 27: 'Na', 28: 'Pa', 29: 'Sa', 30: 'Sha', 31: 'Ta', 32: 'THA', 